In [1]:
!pip install -q transformers accelerate bitsandbytes torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.7 MB/s eta 0:00:00:00:0100:01


In [2]:
import os
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm import tqdm

In [23]:

DATASET_PATH = {
    "reasoning": "/kaggle/input/datasets/vedantbarhate007/prompts/prompts/reasoning_prompts.json",
    "logical": "/kaggle/input/datasets/vedantbarhate007/prompts/prompts/logical_prompts.json",
    "classification": "/kaggle/input/datasets/vedantbarhate007/prompts/prompts/classification_prompts.json",
    "qna": "/kaggle/input/datasets/vedantbarhate007/prompts/prompts/qna_prompts.json"
}

MODELS = {
    "llama":   "meta-llama/Meta-Llama-3-8B-Instruct",
    "mistral": "mistralai/Mistral-7B-Instruct-v0.2",
    "phi":     "microsoft/Phi-3-mini-4k-instruct",
    "qwen": "Qwen/Qwen2.5-1.5B-Instruct",
    "gptoss": "openai/gpt-oss-20b"
}

QUANTIZATION_LEVELS = ["fp16", "8bit", "4bit"]

In [4]:
OUTPUT_ROOT = "/kaggle/working/outputs"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

In [24]:
def load_model(model_name, quant_level):
    print(f"\nLoading {model_name} with {quant_level}...")

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    if quant_level == "fp16":
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16
        )
        model = torch.compile(model)

    elif quant_level == "8bit":
        # Wrap 8-bit logic in BitsAndBytesConfig
        quant_config = BitsAndBytesConfig(load_in_8bit=True)
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=quant_config,
            **model_kwargs
        )

    elif quant_level == "4bit":
        # Wrap 4-bit logic in BitsAndBytesConfig
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=quant_config,
            **model_kwargs
        )

    else:
        raise ValueError("Invalid quantization level. Choose from 'fp16', '8bit', or '4bit'.")

    return tokenizer, model

In [6]:
temp = 0.5
top_p = 0.9
max_token = 384
rep_pen = 1.2

SYSTEM_PROMPT = """You are a knowledgeable assistant. Answer the question directly and concisely.
You are a precise and concise AI assistant.
Rules:
- The response MUST be under 256 tokens.
- Aim for 150–220 tokens when possible.
- Do not exceed 256 tokens under any circumstances.
- No filler, no repetition, no unnecessary explanations.
- Be direct, structured, and information-dense.
- Use short paragraphs or bullet points when helpful.
- Do not restate the question.
- Do not include disclaimers unless explicitly required.
- End immediately after completing the answer.
If the answer would exceed 256 tokens, summarize aggressively to stay within limit.
"""

def generate_response(tokenizer, model, prompt):

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt}
    ]

    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            repetition_penalty=1.2,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    response = outputs[0][input_len:]
    return tokenizer.decode(response, skip_special_tokens=True).strip()


In [7]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_secret = user_secrets.get_secret("HF_TOKEN")


In [8]:
from huggingface_hub import login

login(token=hf_secret)

In [25]:
MODEL = MODELS["phi"]   # llama, mistral, phi, qwen, gptoss
QUANT = QUANTIZATION_LEVELS[0]  #0-fp16, 1-8bit, 2-4bit

tokenizer, model = load_model(MODEL, QUANT)

output_folder = os.path.join(OUTPUT_ROOT, f"{MODEL}_{QUANT}")
os.makedirs(output_folder, exist_ok=True)


Loading microsoft/Phi-3-mini-4k-instruct with fp16...


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [26]:
task_category = "reasoning"     # reasoning, logical, classification, qna


DATASET_PATH = DATASET_PATH[task_category]
with open(DATASET_PATH, "r") as f:
    dataset = json.load(f)

prompts = dataset["prompts"]

print(f"Loaded {len(prompts)} prompts")


Loaded 160 prompts


In [ ]:
output_folder_for_task = os.path.join(output_folder, f"{task_category}")
os.makedirs(output_folder_for_task, exist_ok=True)

checkpoint_path = os.path.join(output_folder_for_task, "responses.jsonl")
final_json_path = os.path.join(output_folder_for_task, "responses.json")

for item in tqdm(prompts):
    prompt_text = item["prompt"]

    try:
        response = generate_response(tokenizer, model, prompt_text)
    except Exception as e:
        response = f"ERROR: {str(e)}"
    print(f"-> {response[:50]}")

    # Save immediately to prevent data loss
    with open(checkpoint_path, "a") as f:
        result = {**item, "model": MODEL, "quantization": QUANT, "response": response}
        f.write(json.dumps(result) + "\n")

print(f"Process complete. Responses saved to {checkpoint_path}")


final_results = []
try:
    with open(checkpoint_path, "r") as f:
        for line in f:
            final_results.append(json.loads(line))

    with open(final_json_path, "w") as f:
        json.dump(final_results, f, indent=4)

    print(f"\n✅ All responses processed and saved to a clean JSON list at: {final_json_path}")

    # Optional: Remove the .jsonl checkpoint if you only want the final JSON
    # os.remove(checkpoint_path)

except Exception as e:
    print(f"Conversion failed, but your data is safe in {checkpoint_path}. Error: {e}")


  0%|          | 0/160 [00:00<?, ?it/s]

In [22]:
# Memory Cleanup
del model
del tokenizer
torch.cuda.empty_cache()

In [21]:
! zip -r outputs.zip /kaggle/working/outputs


updating: kaggle/working/outputs/ (stored 0%)
updating: kaggle/working/outputs/meta-llama/ (stored 0%)
updating: kaggle/working/outputs/meta-llama/Meta-Llama-3-8B-Instruct_fp16/ (stored 0%)
updating: kaggle/working/outputs/meta-llama/Meta-Llama-3-8B-Instruct_fp16/classification/ (stored 0%)
updating: kaggle/working/outputs/meta-llama/Meta-Llama-3-8B-Instruct_fp16/classification/responses.json (deflated 82%)
updating: kaggle/working/outputs/meta-llama/Meta-Llama-3-8B-Instruct_fp16/classification/responses.jsonl (deflated 81%)
  adding: kaggle/working/outputs/meta-llama/Meta-Llama-3-8B-Instruct_fp16/qna/ (stored 0%)
  adding: kaggle/working/outputs/meta-llama/Meta-Llama-3-8B-Instruct_fp16/qna/responses.jsonl (deflated 83%)
  adding: kaggle/working/outputs/meta-llama/Meta-Llama-3-8B-Instruct_fp16/qna/responses.json (deflated 83%)
  adding: kaggle/working/outputs/meta-llama/Meta-Llama-3-8B-Instruct_fp16/logical/ (stored 0%)
  adding: kaggle/working/outputs/meta-llama/Meta-Llama-3-8B-Instru